# Data Ingestion in the HDFS system


0.1 Step 1: Setting Up the Spark Environment
 * In a real-world company setup, we wouldn’t use Google Colab directly. Instead, we would:
 1.  Deploy a Spark Cluster (like AWS EMR, GCP Dataproc, or an on-prem Hadoop cluster, Azure HD Insight).
 2.  Store Data in HDFS instead of local storage.
 • Load data from Kaggle i.e. Data Source (#!/bin/bash curl -L -o ~/olist/brazilianecommerce.zip\ https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilianecommerce)
 • !unzip brazilian-ecommerce.zip -d ~/olist/data/
3.  Use PySpark to interact with data.

# Module 1 - Data Exploration

In [ ]:
spark

In [ ]:
!hadoop fs -ls /data/olist/


In [ ]:
hdfs_path='/data/olist/'

In [ ]:
customers_df=spark.read.csv(hdfs_path + 'olist_customers_dataset.csv',header=True,inferSchema=True)

In [ ]:
customers_df.show(5)

In [ ]:

orders_df = spark.read.csv(hdfs_path + 'olist_orders_dataset.csv',header=True,inferSchema=True)
order_item_df = spark.read.csv(hdfs_path + 'olist_order_items_dataset.csv',header=True,inferSchema=True)
payments_df = spark.read.csv(hdfs_path + 'olist_order_payments_dataset.csv',header=True,inferSchema=True)
reviews_df = spark.read.csv(hdfs_path + 'olist_order_reviews_dataset.csv',header=True,inferSchema=True)
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv',header=True,inferSchema=True)
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv',header=True,inferSchema=True)
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv',header=True,inferSchema=True)
category_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv',header=True,inferSchema=True)


In [ ]:
customers_df.printSchema()

In [ ]:
orders_df.printSchema()

In [ ]:
# Data Leakage or Drop
print(f'customers : {customers_df.count()} rows()')

In [ ]:
print(f'orders : {orders_df.count()} rows()')

In [ ]:
# Null or Duplicate values

from pyspark.sql.functions import col
customers_df.select([col(c).isNull().alias(c) for c in customers_df.columns]).show(5)


In [ ]:
# Check for Nulls in critical fields


from pyspark.sql.functions import col,when,count
customers_df.select([count(when(col(c).isNull(),1)).alias(c) for c in customers_df.columns]).show(10)


In [ ]:
# Check Duplicate Values
customers_df.groupBy('customer_id').count().filter('count>1').show()

In [ ]:
# customer distribution by state
customers_df.groupBy('customer_state').count().orderBy('count',ascending=False).show(5)

In [ ]:
# order - order_status distribuiton
orders_df.groupBy('order_status').count().orderBy('count',ascending = False).show(10)


In [ ]:
# payments
payments_df.show(3)


In [ ]:
payments_df.groupBy('payment_type').count().orderBy(col('count').desc()).show()

In [ ]:
# Top selling items
order_item_df.show(5)

In [ ]:
from pyspark.sql.functions import sum
top_products=order_item_df.groupBy('product_id').agg(sum('price').alias('Total_sale'))
top_products.orderBy('Total_sale',ascending=False).show(5)

In [ ]:
# Average delivery time analysis
delivery_df=orders_df.select('order_id','order_purchase_timestamp','order_delivered_customer_date')

In [ ]:
delivery_df.show(3)

In [ ]:
from pyspark.sql.functions import datediff,to_date
delivery_detail_df=delivery_df.withColumn('delivery_time',datediff(col('order_delivered_customer_date'),col('order_purchase_timestamp')))

In [ ]:
delivery_detail_df.orderBy('delivery_time',ascending=False).show(10)

In [ ]:
spark.stop()